## Natural Language Processing Spring 2026
---
# LoRA Fine-Tuning, DPO, and PPO (RLHF) on MiniMind

**Objective:** Take a *pretrained* and a *Supervised Fine-Tuned* (SFT) MiniMind checkpoint that **you already have on disk** and run the three most common post-training steps used in modern LLM pipelines, end-to-end:

* **LoRA** — parameter-efficient fine-tuning on top of either the pretrained or the SFT base.
* **DPO** — Direct Preference Optimization (an RLHF method) using preference pairs.
* **PPO** — Proximal Policy Optimization (the classical RLHF method) using a learned value head and a KL penalty against a reference model.



To match the official MiniMind training pipeline, every implementation in this notebook is a faithful demo-sized reproduction of the corresponding script under `trainer/`:

| Step | Mirrors | Key file |
|---|---|---|
| LoRA | `apply_lora`, `save_lora`, `load_lora`, `merge_lora` | `model/model_lora.py`, `trainer/train_lora.py` |
| DPO  | `logits_to_log_probs`, `dpo_loss`, `train_epoch` | `trainer/train_dpo.py` |
| PPO  | `CriticModel`, GAE, clipped surrogate, KL-to-ref | `trainer/train_ppo.py` |



The notebook is structured so that:

* **Step 1** downloads the RLHF / SFT JSONL files we need (LoRA + DPO + PPO data).
* **Step 2** is a self-contained recap of the MiniMind micro-architecture.
* **Step 3** **loads** your `pretrain_*.pth` and `full_sft_*.pth` checkpoints from a Colab file path — no retraining and no Hugging Face download. This mirrors how you would run the official `trainer/train_lora.py`, `trainer/train_dpo.py`, or `trainer/train_ppo.py` script in production.
* **Step 4–5** implement and run **LoRA** on the base of your choice (pretrain *or* SFT).
* **Step 6–7** implement and run **DPO** against a frozen reference model.
* **Step 8** implements and runs **PPO** with a critic, GAE advantages, clipped surrogate loss, and a KL-to-reference penalty.
* **Step 9** does a side-by-side comparison of all four checkpoints (SFT, SFT+LoRA, DPO, PPO).



This notebook is a companion to `MiniMindModel.ipynb` (architecture) and `minimind_serious_pretrain_SFT.ipynb` (data + pre-training + SFT). It re-uses the same architecture (`hidden_size=768`, `num_hidden_layers=8`, ~64 M params, MiniMind-3 main branch) so the LoRA/DPO/PPO stages here can load the checkpoints produced by `minimind_serious_pretrain_SFT.ipynb` directly.



### Where this fits in the MiniMind training pipeline

| Phase | Notebook / script | Output checkpoint | Role |
|---|---|---|---|
| 1. Pre-training | `minimind_data_pretrain.ipynb` / `trainer/train_pretrain.py` | `pretrain_{hidden}.pth` | Learn language structure |
| 2. SFT          | `minimind_data_pretrain.ipynb` / `trainer/train_full_sft.py` | `full_sft_{hidden}.pth` | Learn to follow instructions |
| 3a. LoRA        | **this notebook** / `trainer/train_lora.py` | `lora_demo.pth` (+ merged checkpoint) | Lightweight domain/style adaptation |
| 3b. DPO (RLHF)  | **this notebook** / `trainer/train_dpo.py` | `dpo_model.pth` | Align with human preferences (offline) |
| 3c. PPO (RLHF)  | **this notebook** / `trainer/train_ppo.py` | `ppo_model.pth` | Align with human preferences (on-policy) |

The official repo file-naming convention (see `README.md`) is `full_sft_{hidden_size}.pth`, `lora_xxx_{hidden_size}.pth`, `dpo_{hidden_size}.pth`, `ppo_{hidden_size}.pth`; we use shorter demo names below for clarity.

## Step 0: Setup and Imports

### Library Overview

| Import | Role |
|---|---|
| `os`, `json`, `copy` | File system, JSON parsing, deep copies for the frozen reference model |
| `math`, `time`, `random` | Numerics, timing, reproducibility |
| `torch`, `torch.nn`, `torch.nn.functional` | Core tensor ops, layers, and loss functions |
| `torch.optim` | `AdamW` optimiser used by every training stage |
| `torch.utils.data.Dataset` / `DataLoader` | Same `Dataset` contract used in the previous notebook |
| `contextlib.nullcontext` | A no-op context manager so the same `with autocast_ctx:` works on CPU |
| `huggingface_hub.hf_hub_download` | Pulls the MiniMind preference dataset (`dpo.jsonl`) from the Hub |
| `datasets.load_dataset` | Streaming JSONL reader for the preference data |

In [1]:
import os
if not os.path.isdir('/content/minimind-colab'):
    import subprocess
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/Boyu-Zhang-UOI/minimind-colab.git',
         '/content/minimind-colab'],
        check=False,
    )

!pip install -q huggingface_hub datasets transformers

import os
import json
import math
import time
import copy
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext
from datasets import load_dataset, Features, Value
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")


PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4


## Step 1: Download Datasets

We need two JSONL inputs for the post-training stages. We do **not** need any pre-training data here — Step 3 loads your *already-trained* pretrain and SFT checkpoints from disk.

| File | Format | Used by |
|---|---|---|
| `sft_t2t_mini.jsonl` | `{"conversations": [{role, content}, ...]}` | LoRA fine-tuning data (Step 5) and PPO prompt source (Step 8 — same role as `RLAIFDataset` in `dataset/lm_dataset.py`) |
| `dpo.jsonl` | `{"chosen": [...], "rejected": [...]}` | DPO preference data (Step 7) — see `dataset/lm_dataset.py:DPODataset` |



### What is preference data?

Each line of `dpo.jsonl` is a **pair of conversations sharing the same prompt** — one labelled *chosen* (the response a human annotator preferred) and one *rejected*. RLHF algorithms (both DPO and PPO with a learned reward model) consume the **relative ordering** of the two responses rather than absolute reward scores; this is the data shape we will exploit when deriving the DPO loss in Step 7.

```
{
  "chosen":   [{"role": "user", "content": "..."}, {"role": "assistant", "content": "<good answer>"}],
  "rejected": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "<worse answer>"}]
}
```



For PPO the original `trainer/train_ppo.py` uses an *on-policy* RLAIF dataset of prompts only (`dataset/lm_dataset.py:RLAIFDataset`) and queries an external reward model on the rolled-out responses. In Step 8 we reuse the prompts inside `sft_t2t_mini.jsonl` as our RLAIF prompt source and replace the external reward model with a small heuristic reward, so the demo runs in a single Colab session. The training-loop structure (rollout → GAE → clipped surrogate loss + KL penalty) is identical to `trainer/train_ppo.py`.

In [2]:
# Download the JSONL files we need
repo_id = "jingyaogong/minimind_dataset"
files_to_download = [
    "sft_t2t_mini.jsonl",   # LoRA fine-tuning data + PPO prompt source
    "dpo.jsonl",            # DPO preference pairs
]

dataset_dir = "./dataset"
os.makedirs(dataset_dir, exist_ok=True)

print("Downloading datasets from Hugging Face...")
for file_name in files_to_download:
    print(f"Fetching {file_name}...")
    hf_hub_download(
        repo_id=repo_id,
        filename=file_name,
        repo_type="dataset",
        local_dir=dataset_dir,
    )

print("\nAll datasets downloaded to ./dataset.")

Fetching sft_t2t_mini.jsonl...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sft_t2t_mini.jsonl:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Fetching dpo.jsonl...


dpo.jsonl:   0%|          | 0.00/53.7M [00:00<?, ?B/s]


All datasets downloaded to ./dataset.


### Inspecting the preference data

Before writing a single line of training code it is worth eyeballing `dpo.jsonl` carefully — DPO is **only** as good as the contrast between its `chosen` and `rejected` responses, and any structural bug we miss here will silently corrupt training.

Each line of `dpo.jsonl` is a JSON object of the form

```json
{
  "chosen":   [{"role": "user", "content": "..."}, {"role": "assistant", "content": "<good answer>"}],
  "rejected": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "<worse answer>"}]
}
```

The two lists share the user turn (the prompt) and only differ in the assistant turn. The cell below verifies that assumption and gathers the basic statistics that drive every later design choice in this notebook:

* **# of records** → how many gradient signals we have. DPO is sample-efficient but still benefits from more pairs.
* **Always-paired** → every record must have *both* a `chosen` and a `rejected` field, otherwise we cannot form a contrastive pair.
* **Same user prompt** → if `chosen[0].content != rejected[0].content` then we are accidentally comparing answers to *different* questions; the DPO margin would be meaningless.
* **Token-length distribution of the assistant turn** → with the real MiniMind BPE tokenizer (`vocab_size=6400`), this tells us how long `dpo_seq_len` needs to be in Step 6 so we don't truncate the assistant span (where the entire DPO loss mask lives).
* **Chosen vs rejected length** → a strong systematic length bias (e.g. *chosen always longer*) is a classic preference-data smell; see [Singhal et al. 2023](https://arxiv.org/abs/2310.03716). DPO can pick up on length alone instead of quality, so it is worth knowing.

In [4]:
import statistics

dpo_path = "./dataset/dpo.jsonl"

# Load the real BPE tokenizer
# We need it here just for the token-length statistics below.
try:
    _tok = tokenizer  # already defined?
except NameError:
    _tokenizer_dir = '/content/minimind-colab/model'
    if not os.path.isdir(_tokenizer_dir):
        _tokenizer_dir = './model'  # local fallback
    _tok = AutoTokenizer.from_pretrained(_tokenizer_dir)

if os.path.exists(dpo_path):
    n_records          = 0
    n_missing_field    = 0
    n_prompt_mismatch  = 0
    chosen_tok_lens    = []
    rejected_tok_lens  = []
    chosen_longer      = 0
    rejected_longer    = 0
    equal_len          = 0
    first_sample       = None

    with open(dpo_path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if first_sample is None:
                first_sample = rec
            n_records += 1

            chosen   = rec.get("chosen")   or []
            rejected = rec.get("rejected") or []
            if not chosen or not rejected:
                n_missing_field += 1
                continue

            # Same-prompt check — first user turn must match on both sides.
            user_c = next((t.get("content", "") for t in chosen   if t.get("role") == "user"), None)
            user_r = next((t.get("content", "") for t in rejected if t.get("role") == "user"), None)
            if user_c != user_r:
                n_prompt_mismatch += 1

            # Token length of the assistant turn (what DPO will actually score).
            asst_c = " ".join(t.get("content", "") for t in chosen   if t.get("role") == "assistant")
            asst_r = " ".join(t.get("content", "") for t in rejected if t.get("role") == "assistant")
            lc = len(_tok(asst_c, add_special_tokens=False).input_ids)
            lr = len(_tok(asst_r, add_special_tokens=False).input_ids)
            chosen_tok_lens.append(lc)
            rejected_tok_lens.append(lr)
            if   lc > lr: chosen_longer   += 1
            elif lr > lc: rejected_longer += 1
            else:         equal_len       += 1

    print(f"dpo.jsonl: {n_records:,} preference pairs")
    print(f"  missing chosen/rejected field : {n_missing_field}")
    print(f"  user-prompt mismatch          : {n_prompt_mismatch}")
    print()
    def _stats(name, xs):
        if not xs: return
        xs_sorted = sorted(xs)
        p50 = xs_sorted[len(xs_sorted)//2]
        p95 = xs_sorted[int(0.95*len(xs_sorted))]
        print(f"  {name:18s}: min={min(xs):4d}  median={p50:4d}  mean={statistics.mean(xs):6.1f}  p95={p95:4d}  max={max(xs):4d}")
    _stats("chosen   tok-len", chosen_tok_lens)
    _stats("rejected tok-len", rejected_tok_lens)
    print()
    print(f"  chosen longer  : {chosen_longer:5d}  ({100*chosen_longer / max(1,n_records):.1f}%)")
    print(f"  rejected longer: {rejected_longer:5d}  ({100*rejected_longer / max(1,n_records):.1f}%)")
    print(f"  equal length   : {equal_len:5d}  ({100*equal_len / max(1,n_records):.1f}%)")

    print("\nExample record (first 2000 chars):")
    print(json.dumps(first_sample, ensure_ascii=False, indent=2)[:2000])
else:
    print(f"{dpo_path} not found — re-run the download cell above.")


dpo.jsonl: 17,166 preference pairs
  missing chosen/rejected field : 0
  user-prompt mismatch          : 0

  chosen   tok-len  : min=   1  median= 294  mean= 342.8  p95= 825  max=1239
  rejected tok-len  : min=   1  median= 248  mean= 315.7  p95= 819  max=1283

  chosen longer  :  9070  (52.8%)
  rejected longer:  7992  (46.6%)
  equal length   :   104  (0.6%)

Example record (first 2000 chars):
{
  "chosen": [
    {
      "content": "continue",
      "role": "user"
    },
    {
      "content": "As Recharge Retreats grows, we plan to expand our team with additional event coordinators and marketing specialists. To accommodate this growth, we will establish an office space that fosters collaboration and creativity among team members. We will also prioritize remote work options and streamlined communication tools to support remote team members.\n\nIn addition, we aim to diversify our retreat offerings to cater to different themes, such as solo travel retreats, wellness retreats for pare

**Reading the output above:**

* **Pair count.** Every line is one preference pair, i.e. one DPO gradient signal. DPO is sample-efficient compared with PPO (no rollouts, no reward model), but more pairs still help — especially because we *only* supervise the assistant tokens, so the effective number of supervised tokens is quite a bit smaller than for SFT.
* **No missing fields, no prompt mismatch.** These two checks must read `0`. If they don't, the offending lines need to be filtered out before training, otherwise the DPO loss becomes a comparison between *unrelated* responses (the contrastive signal disappears).
* **Token-length distribution.** This is what determines the `max_length` we pass into `DPODataset` in Step 6. The full chat string we tokenise has the form `<bos><user>\n<prompt><assistant>\n<chosen|rejected><eos>`, so we need at least `prompt_tokens + assistant_tokens + ~5` to keep the assistant span (and therefore the DPO loss mask) intact. Picking `max_length` *below* the assistant p95 means most of the loss mask is truncated to zero — and the model never sees a useful gradient. We use this to size `dpo_seq_len` below.
* **Chosen-vs-rejected length skew.** A heavy bias here is a known DPO failure mode: the model can learn to prefer longer responses regardless of quality. The split should be roughly balanced (or the rejected set should be slightly longer); a one-sided skew is worth flagging.
* **Example record.** Confirms the schema: two lists of `{role, content}` dicts that share the user turn and differ only in the assistant turn. This is exactly the structure `DPODataset` expects in Step 6.

## Step 2: MiniMind Architecture (Standalone Recap)

We re-include the same micro-architecture used in `MiniMindModel.ipynb` and `minimind_data_pretrain.ipynb` so this notebook is fully self-contained. The forward signature

```python
model(input_ids, position_embeddings, labels=None) -> (logits, loss)
```

is identical to the previous notebooks, so the LoRA / DPO trainers below can plug in directly.

The recap is split into three cells:
1. **Configuration & Normalization** — `MiniMindConfig` and `RMSNorm`.
2. **Position Embeddings** — `precompute_freqs_cis` and `apply_rotary_pos_emb` (RoPE).
3. **Attention, FeedForward, Block, and the Causal LM** — the rest of the Transformer.

In [5]:
# 1. Configuration & Normalization
class MiniMindConfig:
    """Hyperparameter container — kept tiny for Colab."""
    def __init__(self):
        self.vocab_size = 6400
        self.hidden_size = 768
        self.num_hidden_layers = 8
        self.num_attention_heads = 8
        self.num_key_value_heads = 4
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.intermediate_size = 2432
        self.max_position_embeddings = 2048
        self.rms_norm_eps = 1e-6
        self.dropout = 0.0
        self.tie_word_embeddings = True

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (self.weight * normed.float()).type_as(x)

In [6]:
# 2. Rotary Position Embeddings (RoPE)
def precompute_freqs_cis(dim: int, end: int, rope_base: float = 1e6):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin

def apply_rotary_pos_emb(q, k, cos, sin):
    def rotate_half(x):
        half_dim = x.shape[-1] // 2
        return torch.cat((-x[..., half_dim:], x[..., :half_dim]), dim=-1)
    cos_u = cos.unsqueeze(0).unsqueeze(2)
    sin_u = sin.unsqueeze(0).unsqueeze(2)
    return (q * cos_u) + (rotate_half(q) * sin_u), (k * cos_u) + (rotate_half(k) * sin_u)

In [7]:
# 3. Attention, FeedForward, Transformer Block, and Causal LM
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    bs, slen, num_kv_heads, head_dim = x.shape
    if n_rep == 1:
        return x
    return x[:, :, :, None, :].expand(bs, slen, num_kv_heads, n_rep, head_dim).reshape(bs, slen, num_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = config.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, self.n_local_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.n_local_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
    def forward(self, x, position_embeddings):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        xq = xq.transpose(1, 2)
        xk = repeat_kv(xk, self.n_rep).transpose(1, 2)
        xv = repeat_kv(xv, self.n_rep).transpose(1, 2)
        scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
        scores = scores + mask
        attention_output = F.softmax(scores, dim=-1) @ xv
        attention_output = attention_output.transpose(1, 2).contiguous().reshape(bsz, seq_len, -1)
        return self.o_proj(attention_output)

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj   = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class MiniMindBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size)
        self.post_attention_layernorm = RMSNorm(config.hidden_size)
        self.mlp = FeedForward(config)
    def forward(self, x, position_embeddings):
        x = x + self.self_attn(self.input_layernorm(x), position_embeddings)
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

class MiniMindModel(nn.Module):
    """Inner backbone — matches model/model_minimind.py:MiniMindModel so the
    state_dict keys produced by `minimind_serious_pretrain_SFT.ipynb`
    (`model.embed_tokens.weight`, `model.layers.X.…`, `model.norm.weight`)
    load directly into this notebook's models."""
    def __init__(self, config):
        super().__init__()
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([MiniMindBlock(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size)
    def forward(self, input_ids, position_embeddings):
        x = self.embed_tokens(input_ids)
        for layer in self.layers:
            x = layer(x, position_embeddings)
        return self.norm(x)

class MiniMindForCausalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.model = MiniMindModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        if config.tie_word_embeddings:
            self.model.embed_tokens.weight = self.lm_head.weight
    def forward(self, input_ids, position_embeddings, labels=None):
        x = self.model(input_ids, position_embeddings)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100)
        return logits, loss


## Step 3: Load Pretrained and SFT Checkpoints from a Colab File Path

In production you train the pretrain and SFT checkpoints with `trainer/train_pretrain.py` and `trainer/train_full_sft.py`, then point the post-training scripts (`train_lora.py`, `train_dpo.py`, `train_ppo.py`) at those `.pth` files via the `--model_path` flag. This notebook does the same: **no retraining and no Hugging Face download** — instead, you provide a Colab file path to two checkpoints that match the architecture defined in Step 2.



### Where do these `.pth` files come from?

Pick whichever is most convenient:

* **Hugging Face**, e.g. `hf_hub_download("jingyaogong/minimind-3-pytorch", "full_sft_768.pth", local_dir="./checkpoints")`. (You only need to do this once per session.)
* **Google Drive** — `from google.colab import drive; drive.mount('/content/drive')` and then point `SFT_CKPT_PATH` at `/content/drive/MyDrive/...`.
* **Direct upload** — drag-and-drop a `.pth` into the Colab file pane (`Files → Upload`) and set the path to e.g. `/content/full_sft_768.pth`.
* **Demo checkpoints** — if you have already run `minimind_data_pretrain.ipynb` in the same session, the files at `./checkpoints/pretrain_model.pth` and `./checkpoints/sft_model.pth` will work as-is.

> ⚠️ **Architecture must match.** The state dict you load is keyed by parameter name and shape. The official release targets `hidden_size=768, num_hidden_layers=8`, and this notebook's `MiniMindConfig` is set to match (so checkpoints produced by `minimind_serious_pretrain_SFT.ipynb` load directly). If you point at a checkpoint whose shapes do not match `MiniMindConfig` from Step 2, `load_state_dict` will raise a shape-mismatch error — adjust `MiniMindConfig` to match your file in that case.

### Configure paths and shared training state

The two cells below are the **only** thing you should ever need to edit when re-running this notebook against a new pair of checkpoints. They mirror the `--model_path` argument of `trainer/train_lora.py`, `trainer/train_dpo.py`, and `trainer/train_ppo.py`.

In [10]:
class PretrainDataset(Dataset):
    """Same definition as in minimind_data_pretrain.ipynb / dataset/lm_dataset.py."""
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = load_dataset('json', data_files=data_path, split='train')
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, index):
        sample = self.samples[index]
        tokens = self.tokenizer(str(sample['text']), add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))
        input_ids = torch.tensor(input_ids, dtype=torch.long)
        labels = input_ids.clone()
        labels[input_ids == self.tokenizer.pad_token_id] = -100
        return input_ids, labels

class SFTDataset(Dataset):
    """Same SFT logic as dataset/lm_dataset.py:SFTDataset — masks every non-assistant token."""
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        features = Features({'conversations': [{'role': Value('string'), 'content': Value('string'), 'reasoning_content': Value('string'), 'tools': Value('string'), 'tool_calls': Value('string')}]})
        self.samples = load_dataset('json', data_files=jsonl_path, split='train', features=features)
    def __len__(self):
        return len(self.samples)
    def _encode(self, text):
        return self.tokenizer(text, add_special_tokens=False, max_length=self.max_length, truncation=False).input_ids
    def __getitem__(self, index):
        sample = self.samples[index]
        conversations = sample['conversations']
        input_ids = [self.tokenizer.bos_token_id]
        labels = [-100]
        for turn in conversations:
            role = (turn.get('role') or 'user')
            content = str(turn.get('content') or '')
            header_ids = self._encode(f"<{role}>\n")
            content_ids = self._encode(content)
            input_ids += header_ids + content_ids
            if role == 'assistant':
                labels += [-100] * len(header_ids) + list(content_ids)
            else:
                labels += [-100] * (len(header_ids) + len(content_ids))
        input_ids.append(self.tokenizer.eos_token_id)
        if len(labels) and labels[-1] != -100:
            labels.append(self.tokenizer.eos_token_id)
        else:
            labels.append(-100)
        input_ids = input_ids[:self.max_length]
        labels = labels[:self.max_length]
        pad_len = self.max_length - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        labels += [-100] * pad_len
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

In [9]:
# === EDIT THESE TWO PATHS ===========================================
PRETRAIN_CKPT_PATH = "/content/checkpoints/pretrain_768.pth"   # output of trainer/train_pretrain.py
SFT_CKPT_PATH      = "/content/checkpoints/full_sft_768.pth"   # output of trainer/train_full_sft.py
# ====================================================================

In [11]:
class TrainArgs:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    epochs = 3
    sft_learning_rate = 5e-5
    accumulation_steps = 2
    grad_clip = 1.0
    log_interval = 5
    max_seq_len = 32

args = TrainArgs()
config = MiniMindConfig()
os.makedirs("./checkpoints", exist_ok=True)

In [12]:
# Load the real MiniMind BPE tokenizer (vocab_size=6400)
# The tokenizer files ship in the `model/` folder of the cloned repo.
TOKENIZER_DIR = '/content/minimind-colab/model'
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
config.vocab_size = tokenizer.vocab_size
print(f"Tokenizer vocab size : {tokenizer.vocab_size}")
print(f"BOS / EOS / PAD ids  : {tokenizer.bos_token_id} / {tokenizer.eos_token_id} / {tokenizer.pad_token_id}")

pos_embs = precompute_freqs_cis(config.head_dim, args.max_seq_len)
pos_embs = (pos_embs[0].to(args.device), pos_embs[1].to(args.device))


Tokenizer vocab size : 6400
BOS / EOS / PAD ids  : 1 / 2 / 0
Loading pretrain checkpoint : /content/checkpoints/pretrain_768.pth
Loading SFT      checkpoint : /content/checkpoints/full_sft_768.pth
Pretrain logits shape: (2, 32, 6400)
SFT      logits shape: (2, 32, 6400)
Both checkpoints loaded successfully.


In [13]:
# Lightweight helper — used by every subsequent step that needs to load a base.
def load_minimind_checkpoint(path: str) -> "MiniMindForCausalLM":
    """Build a fresh model and load a state-dict from `path`.

    Mirrors the `init_model` logic at the top of every training script under
    `trainer/`: build the architecture, then `model.load_state_dict(...)`.

    Uses `strict=False` so a checkpoint produced by the production
    `MiniMindForCausalLM` (which may carry extra keys like `freqs_cos`/`freqs_sin`
    rotary buffers that this notebook computes inline) loads cleanly.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Checkpoint not found: {path}. Set PRETRAIN_CKPT_PATH / SFT_CKPT_PATH to a valid "
            ".pth file (Hugging Face download, Google Drive mount, or a manual upload)."
        )
    m = MiniMindForCausalLM(config).to(args.device)
    state = torch.load(path, weights_only=True, map_location=args.device)
    missing, unexpected = m.load_state_dict(state, strict=False)
    if missing:
        print(f"  [warn] {len(missing)} missing key(s), e.g. {missing[:3]}")
    if unexpected:
        print(f"  [warn] {len(unexpected)} unexpected key(s), e.g. {unexpected[:3]}")
    return m

In [14]:
# Pre-load both checkpoints once and run a sanity forward pass on each.
print("Loading pretrain checkpoint :", PRETRAIN_CKPT_PATH)
model_pretrain = load_minimind_checkpoint(PRETRAIN_CKPT_PATH)
print("Loading SFT      checkpoint :", SFT_CKPT_PATH)
model_sft      = load_minimind_checkpoint(SFT_CKPT_PATH)

with torch.no_grad():
    dummy = torch.randint(0, config.vocab_size, (2, args.max_seq_len), device=args.device)
    pre_logits, _ = model_pretrain(dummy, position_embeddings=pos_embs)
    sft_logits, _ = model_sft(dummy, position_embeddings=pos_embs)
print(f"Pretrain logits shape: {tuple(pre_logits.shape)}")
print(f"SFT      logits shape: {tuple(sft_logits.shape)}")
print("Both checkpoints loaded successfully.")

Loading pretrain checkpoint : /content/checkpoints/pretrain_768.pth
Loading SFT      checkpoint : /content/checkpoints/full_sft_768.pth
Pretrain logits shape: (2, 32, 6400)
SFT      logits shape: (2, 32, 6400)
Both checkpoints loaded successfully.


## Step 4: LoRA — Low-Rank Adaptation



### Theory & Concepts

[LoRA (Hu et al. 2021)](https://arxiv.org/abs/2106.09685) is a parameter-efficient fine-tuning (PEFT) method built on a single empirical observation: when you fine-tune a pre-trained Transformer on a downstream task, the **change** in each weight matrix has a much smaller *intrinsic rank* than the matrix itself. So instead of training the full update $\Delta W \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, we factor it as the product of two thin matrices

$$W' = W + \Delta W = W + B A,\quad A \in \mathbb{R}^{r \times d_{\text{in}}},\; B \in \mathbb{R}^{d_{\text{out}} \times r},\quad r \ll \min(d_{\text{in}}, d_{\text{out}})$$

The base weight $W$ is **frozen**; only $A$ and $B$ are trained.



#### Why this is a big deal

| Quantity | Full fine-tune | LoRA (rank $r$) |
|---|---|---|
| Trainable params per linear | $d_{\text{out}} \cdot d_{\text{in}}$ | $r\,(d_{\text{out}} + d_{\text{in}})$ |
| Example: $1024 \times 1024$, $r=8$ | 1,048,576 | 16,384 (≈64× fewer) |
| Optimizer state (AdamW: 2 floats / param) | $\approx 8\,\text{MB}$ | $\approx 0.13\,\text{MB}$ |
| Checkpoint size | full model | tiny, often a few MB total |

Because the base model is shared, you can train **many** LoRA adapters (one per domain, customer, persona, …) and swap them at inference time without keeping multiple copies of the multi-GB base weights.



#### Two subtle but important details from the reference code

* **Initialisation:** $A$ is sampled from $ \mathscr{N}(0, 0.02^2)$ while $B$ starts at **zero**, so at training step 0 we have $\Delta W = BA = 0$ — the LoRA-augmented model is *bit-identical* to the frozen base model. Without this, training would start from a random perturbation and immediately destroy SFT behaviour. Crucially, because $B = 0$ but its gradient is generically non-zero ($\nabla_B \mathscr{L} \propto A x \cdot \text{grad}_y$), the first optimiser step is well-defined and asymmetry breaks naturally.


* **Square-only filter:** `apply_lora` skips non-square `nn.Linear` modules (those with `weight.shape[0] != weight.shape[1]`). In MiniMind this means LoRA is added to `q_proj`, `o_proj`, `lm_head`, and the FFN's square-only projections; `k_proj`/`v_proj` (which are smaller because of GQA) and the rectangular FFN projections are skipped. This matches the reference `model/model_lora.py` behaviour exactly.



### `save_lora` / `load_lora` / `merge_lora` API

| Helper | What it does | When to use it |
|---|---|---|
| `save_lora(model, path)` | Walks the model, picks out every `module.lora`, and writes only the `A`/`B` weights | After training — produces a *tiny* checkpoint (a few MB instead of the full model size) |
| `load_lora(model, path)` | Scans the saved state dict and routes each `A`/`B` tensor back to the matching `module.lora` | At inference time, on top of a fresh base model that has already been wrapped with `apply_lora` |
| `merge_lora(model, lora_path, save_path)` | Collapses LoRA into base weights via $W \leftarrow W + BA$ and saves a *single* full checkpoint | When you want to deploy without keeping the LoRA wrapper (one-time cost, full checkpoint size) |

#### The `LoRA` module

A textbook implementation: two `nn.Linear` layers, no bias, with the carefully chosen initialisation discussed above.

In [15]:
# ---- model/model_lora.py (verbatim semantics) ---------------------------
class LoRA(nn.Module):
    def __init__(self, in_features, out_features, rank):
        super().__init__()
        self.rank = rank
        self.A = nn.Linear(in_features, rank, bias=False)   # low-rank A
        self.B = nn.Linear(rank, out_features, bias=False)  # low-rank B
        self.A.weight.data.normal_(mean=0.0, std=0.02)      # Gaussian init for A
        self.B.weight.data.zero_()                          # Zero init for B → ΔW = 0 at step 0
    def forward(self, x):
        return self.B(self.A(x))

#### Attaching LoRA to a model: `apply_lora`

We monkey-patch every square `nn.Linear` to call its original forward *plus* the LoRA branch. The `default-arg` trick (`layer1=original_forward, layer2=lora`) avoids Python's notorious **late-binding closure** bug — without it every patched layer would end up referencing the same module that happens to be in scope at the end of the loop.

In [16]:
def apply_lora(model, rank=8):
    """Attach a LoRA branch to every square nn.Linear in the model.
    Mirrors model/model_lora.py:apply_lora — only the device lookup is adapted to
    the demo model (the production model exposes a `.device` attribute).
    """
    device = next(model.parameters()).device
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and module.weight.shape[0] == module.weight.shape[1]:
            lora = LoRA(module.weight.shape[0], module.weight.shape[1], rank=rank).to(device)
            setattr(module, "lora", lora)
            original_forward = module.forward
            # Late-binding pitfall: capture both layers as default args.
            def forward_with_lora(x, layer1=original_forward, layer2=lora):
                return layer1(x) + layer2(x)
            module.forward = forward_with_lora

#### Saving, loading, and merging LoRA weights

In [17]:
def save_lora(model, path):
    raw_model = getattr(model, '_orig_mod', model)
    state_dict = {}
    for name, module in raw_model.named_modules():
        if hasattr(module, 'lora'):
            clean_name = name[7:] if name.startswith("module.") else name
            lora_state = {f'{clean_name}.lora.{k}': v.cpu().half() for k, v in module.lora.state_dict().items()}
            state_dict.update(lora_state)
    torch.save(state_dict, path)

def load_lora(model, path):
    device = next(model.parameters()).device
    state_dict = torch.load(path, map_location=device)
    state_dict = {(k[7:] if k.startswith('module.') else k): v for k, v in state_dict.items()}
    for name, module in model.named_modules():
        if hasattr(module, 'lora'):
            lora_state = {k.replace(f'{name}.lora.', ''): v for k, v in state_dict.items() if f'{name}.lora.' in k}
            module.lora.load_state_dict(lora_state)

def merge_lora(model, lora_path, save_path):
    """Merge LoRA into base weights and save a single checkpoint (W ← W + BA)."""
    load_lora(model, lora_path)
    raw_model = getattr(model, '_orig_mod', model)
    state_dict = {k: v.cpu().half() for k, v in raw_model.state_dict().items() if '.lora.' not in k}
    for name, module in raw_model.named_modules():
        if isinstance(module, nn.Linear) and '.lora.' not in name:
            state_dict[f'{name}.weight'] = module.weight.data.clone().cpu().half()
            if hasattr(module, 'lora'):
                state_dict[f'{name}.weight'] += (module.lora.B.weight.data @ module.lora.A.weight.data).cpu().half()
    torch.save(state_dict, save_path)

#### Sanity check — does `apply_lora` preserve the model at step 0?

Because $B = 0$, attaching LoRA must leave the model output unchanged on the very first forward pass. We verify this on a small dummy batch.

In [18]:
# --- UNIT TEST: ΔW = 0 at initialisation ---
sanity_model = MiniMindForCausalLM(config).to(args.device)
dummy_x = torch.randint(0, config.vocab_size, (2, args.max_seq_len), device=args.device)
sanity_pe = (pos_embs[0][:args.max_seq_len], pos_embs[1][:args.max_seq_len])
with torch.no_grad():
    out_before, _ = sanity_model(dummy_x, position_embeddings=sanity_pe)
    apply_lora(sanity_model, rank=8)
    out_after,  _ = sanity_model(dummy_x, position_embeddings=sanity_pe)

n_lora = sum(p.numel() for n, p in sanity_model.named_parameters() if 'lora' in n)
n_total = sum(p.numel() for p in sanity_model.parameters())
print(f"Total params      : {n_total/1e6:.3f} M")
print(f"LoRA params       : {n_lora/1e6:.3f} M  ({n_lora/n_total*100:.2f}%)")
print(f"Max output diff   : {(out_after - out_before).abs().max().item():.2e}")
assert torch.allclose(out_before, out_after), "LoRA changed the model output at init — B should be zero!"
print("LoRA initialisation is faithful (ΔW = 0 at step 0).")
del sanity_model

Total params      : 64.109 M
LoRA params       : 0.197 M  (0.31%)
Max output diff   : 0.00e+00
LoRA initialisation is faithful (ΔW = 0 at step 0).


## Step 5: LoRA Fine-Tuning Loop

This mirrors `trainer/train_lora.py`. You can attach LoRA to **either** the pretrained or the SFT base — both are valid starting points and are widely used in practice:

| `LORA_BASE` | Starting point | What you typically use it for |
|---|---|---|
| `"sft"` | The SFT checkpoint | The standard recipe — instruction-tune first, then specialise with LoRA on a domain dataset (`lora_medical.jsonl`, `lora_identity.jsonl`, …). This is what `trainer/train_lora.py` does by default (`--model_path full_sft_{hidden}.pth`). |
| `"pretrain"` | The pretrained-only checkpoint | Useful when you want LoRA itself to perform the SFT-like adaptation (e.g. for a specialised assistant where you intend to *replace* SFT) or when no SFT model is available. The training data is still SFT-format conversations. |



The recipe is:

1. **Load the chosen base checkpoint** into a fresh model (using `load_minimind_checkpoint`).
2. **Call `apply_lora`** to attach LoRA branches to every square `nn.Linear`.
3. **Set `requires_grad = True`** only on parameters whose name contains `'lora'`; freeze everything else (`trainer/train_lora.py:138-144`).
4. **Build an `AdamW`** over the LoRA parameters only — `model.parameters()` would silently waste memory on the frozen base weights.
5. **Train on SFT-format data** (the same `SFTDataset` defined in Step 3 — in production this would be `lora_medical.jsonl`, `lora_identity.jsonl`, etc.).
6. **Save** with `save_lora` (LoRA-only weights, a few MB) and **optionally `merge_lora`** for inference.



Notice that the loss is the standard cross-entropy *on top of the SFT objective* — LoRA does not change the loss, only the parameter set being optimised.

> The original `train_lora.py` also adds `res.aux_loss` (used by the MoE variant). Our dense demo model has no MoE, so there is no auxiliary loss term to add.

### Step 5a: Load chosen base (pretrain or SFT), attach LoRA, freeze base

In [19]:
# 0. Choose the LoRA starting point: "sft" (default) or "pretrain".
LORA_BASE = "sft"   # set to "pretrain" to attach LoRA on top of the pretrained-only checkpoint
assert LORA_BASE in {"sft", "pretrain"}
base_path = SFT_CKPT_PATH if LORA_BASE == "sft" else PRETRAIN_CKPT_PATH
print(f"LoRA base: {LORA_BASE!r}  ({base_path})")

# 1. Fresh model + load the chosen base weights via the helper from Step 3.
model_lora = load_minimind_checkpoint(base_path)

# 2. Attach LoRA branches (rank=8 — same default as MiniMind's medical/identity LoRAs).
apply_lora(model_lora, rank=8)

# 3. Freeze everything except LoRA. Mirrors trainer/train_lora.py:138-144.
lora_params = []
for name, param in model_lora.named_parameters():
    if 'lora' in name:
        param.requires_grad = True
        lora_params.append(param)
    else:
        param.requires_grad = False

total_params = sum(p.numel() for p in model_lora.parameters())
lora_count   = sum(p.numel() for p in lora_params)
print(f"Total params : {total_params/1e6:.3f} M")
print(f"LoRA params  : {lora_count/1e6:.3f} M  ({lora_count/total_params*100:.2f}%)")
print("Only LoRA params have requires_grad=True; the rest of the model is frozen.")

LoRA base: 'sft'  (/content/checkpoints/full_sft_768.pth)
Total params : 64.109 M
LoRA params  : 0.197 M  (0.31%)
Only LoRA params have requires_grad=True; the rest of the model is frozen.


### Step 5b: Train the LoRA adapter

Same training loop as Step 3, but every gradient operation (clipping included) operates on `lora_params` only — exactly like `trainer/train_lora.py:44`. The base weights are left untouched.

In [20]:
# 4. Build the SFT-format loader for LoRA training. In production you
#    would point this at a domain-specific dataset (`lora_medical.jsonl`, …).
ds_sft = SFTDataset("./dataset/sft_t2t_mini.jsonl", tokenizer, max_length=args.max_seq_len)
ds_sft.samples = ds_sft.samples.select(range(min(2000, len(ds_sft.samples))))
loader_lora = DataLoader(ds_sft, batch_size=4, shuffle=True)

# 5. AdamW over the LoRA parameters only.
opt_lora = optim.AdamW(lora_params, lr=1e-4)
scaler_lora = torch.amp.GradScaler("cuda", enabled=("cuda" in args.device))

# 6. Reuse the standard training loop. Gradient clipping must operate on the
#    LoRA params only, exactly like trainer/train_lora.py:44.
device_type = "cuda" if "cuda" in args.device else "cpu"
autocast_ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast('cuda', dtype=torch.float16)
model_lora.train()
print("\n>>> Starting Phase: LoRA Fine-Tuning <<<")
for epoch in range(1, args.epochs + 1):
    last_step = 0
    for step, (input_ids, labels) in enumerate(loader_lora, start=1):
        last_step = step
        input_ids, labels = input_ids.to(args.device), labels.to(args.device)
        with autocast_ctx:
            logits, loss = model_lora(input_ids, position_embeddings=pos_embs, labels=labels)
            loss = loss / args.accumulation_steps
        scaler_lora.scale(loss).backward()
        if step % args.accumulation_steps == 0:
            scaler_lora.unscale_(opt_lora)
            torch.nn.utils.clip_grad_norm_(lora_params, args.grad_clip)
            scaler_lora.step(opt_lora); scaler_lora.update()
            opt_lora.zero_grad(set_to_none=True)
        if step % args.log_interval == 0:
            print(f"[LoRA] Epoch {epoch}/{args.epochs} Step {step}: Loss = {loss.item()*args.accumulation_steps:.4f}")
    if last_step and last_step % args.accumulation_steps != 0:
        scaler_lora.unscale_(opt_lora)
        torch.nn.utils.clip_grad_norm_(lora_params, args.grad_clip)
        scaler_lora.step(opt_lora); scaler_lora.update()
        opt_lora.zero_grad(set_to_none=True)

Generating train split: 0 examples [00:00, ? examples/s]


>>> Starting Phase: LoRA Fine-Tuning <<<
[LoRA] Epoch 1/3 Step 5: Loss = 0.8520
[LoRA] Epoch 1/3 Step 10: Loss = 1.0178
[LoRA] Epoch 1/3 Step 15: Loss = 0.6707
[LoRA] Epoch 1/3 Step 20: Loss = 1.9688
[LoRA] Epoch 1/3 Step 25: Loss = 1.1137
[LoRA] Epoch 1/3 Step 30: Loss = 0.9541
[LoRA] Epoch 1/3 Step 35: Loss = 0.6103
[LoRA] Epoch 1/3 Step 40: Loss = 0.5586
[LoRA] Epoch 1/3 Step 45: Loss = 0.6824
[LoRA] Epoch 1/3 Step 50: Loss = 0.6664
[LoRA] Epoch 1/3 Step 55: Loss = 0.6842
[LoRA] Epoch 1/3 Step 60: Loss = 0.5335
[LoRA] Epoch 1/3 Step 65: Loss = 1.2590
[LoRA] Epoch 1/3 Step 70: Loss = 1.0067
[LoRA] Epoch 1/3 Step 75: Loss = 0.3320
[LoRA] Epoch 1/3 Step 80: Loss = 1.5285
[LoRA] Epoch 1/3 Step 85: Loss = 2.0670
[LoRA] Epoch 1/3 Step 90: Loss = 0.1621
[LoRA] Epoch 1/3 Step 95: Loss = 0.3906
[LoRA] Epoch 1/3 Step 100: Loss = 0.5570
[LoRA] Epoch 1/3 Step 105: Loss = 0.8876
[LoRA] Epoch 1/3 Step 110: Loss = 0.6588
[LoRA] Epoch 1/3 Step 115: Loss = 1.3894
[LoRA] Epoch 1/3 Step 120: Loss = 0

### Step 5c: Save the LoRA adapter and produce a merged checkpoint

* `save_lora` writes only the trained `A`/`B` weights (a few MB).
* `merge_lora` consumes that file plus a fresh copy of the base model and writes a *full* checkpoint with $W \leftarrow W + BA$ baked in — handy for plugging straight into our standard `MiniMindForCausalLM(config)` without `apply_lora`.

In [21]:
# 7. Save LoRA-only weights and a merged full checkpoint for evaluation.
save_lora(model_lora, "./checkpoints/lora_demo.pth")
print("Saved LoRA-only weights to ./checkpoints/lora_demo.pth")

# Build a *fresh* base model (same base as we trained on top of) + merge LoRA into
# it, leaving model_lora untouched.
merge_base = load_minimind_checkpoint(base_path)
apply_lora(merge_base, rank=8)
merge_lora(merge_base, "./checkpoints/lora_demo.pth", "./checkpoints/sft_lora_merged.pth")
print("Saved merged LoRA checkpoint to ./checkpoints/sft_lora_merged.pth")

Saved LoRA-only weights to ./checkpoints/lora_demo.pth
Saved merged LoRA checkpoint to ./checkpoints/sft_lora_merged.pth


## Step 6: DPO Preference Dataset

### Theory: what is preference data?

Classical SFT teaches the model to imitate a single "gold" response per prompt. **Preference data** instead encodes a *relative judgment*: for the same prompt $x$, a human annotator was shown two model responses and chose one. We write this as a triple $(x, y_w, y_l)$ where $y_w$ is the winning (chosen) response and $y_l$ is the losing (rejected) one.

Why is this useful?

* It captures things humans struggle to write from scratch (good *style*, lack of hallucination, helpfulness, harmlessness) but find easy to *compare*.
* It is much cheaper to collect than absolute reward annotations.
* Algorithms like PPO and DPO are designed to consume exactly this format.



### Implementation: outputs from `DPODataset`

The implementation below is the demo equivalent of `dataset/lm_dataset.py:DPODataset`. Each record contains two role-formatted conversations — `chosen` and `rejected` — that share the same user prompt. We need three tensors per side:

| Field | Shape | What it is |
|---|---|---|
| `x_chosen`, `x_rejected` | `(seq_len-1,)` | Input tokens (input shifted left, suitable for autoregressive LM) |
| `y_chosen`, `y_rejected` | `(seq_len-1,)` | Target tokens (input shifted right) |
| `mask_chosen`, `mask_rejected` | `(seq_len-1,)` | 1 where the target falls inside an *assistant* turn, 0 otherwise — used by the DPO loss to ignore prompt tokens |

The original code uses `tokenizer.apply_chat_template` and scans for `<bos>assistant\n ... <eos>\n` sentinels. Here we build the chat string manually with the same `<role>\n` markers as the SFT data pipeline (so we don't depend on a chat template being registered on the tokenizer) and mark every assistant span as supervised — the **exact same convention** used by `SFTDataset` in Step 3.

In [22]:
class DPODataset(Dataset):
    """Demo DPO dataset — same outputs as dataset/lm_dataset.py:DPODataset."""
    def __init__(self, jsonl_path, tokenizer, max_length=128):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = load_dataset('json', data_files=jsonl_path, split='train')

    def __len__(self):
        return len(self.samples)

    def _encode_pair(self, conversation):
        """Tokenise one (chosen|rejected) conversation and produce input_ids + loss mask.
        loss_mask[i] == 1 iff token i is part of an assistant turn (the only positions DPO supervises).
        """
        input_ids = [self.tokenizer.bos_token_id]
        loss_mask = [0]
        for turn in conversation:
            role = (turn.get('role') or 'user')
            content = str(turn.get('content') or '')
            header_ids  = self.tokenizer(f"<{role}>\n", add_special_tokens=False, max_length=self.max_length, truncation=True).input_ids
            content_ids = self.tokenizer(content,        add_special_tokens=False, max_length=self.max_length, truncation=True).input_ids
            input_ids += header_ids + content_ids
            if role == 'assistant':
                loss_mask += [0] * len(header_ids) + [1] * len(content_ids)
            else:
                loss_mask += [0] * (len(header_ids) + len(content_ids))
        input_ids.append(self.tokenizer.eos_token_id)
        loss_mask.append(1 if (len(loss_mask) and loss_mask[-1] == 1) else 0)
        # Truncate / pad.
        input_ids = input_ids[:self.max_length]
        loss_mask = loss_mask[:self.max_length]
        pad_len = self.max_length - len(input_ids)
        input_ids += [self.tokenizer.pad_token_id] * pad_len
        loss_mask += [0] * pad_len
        return input_ids, loss_mask

    def __getitem__(self, index):
        sample = self.samples[index]
        chosen_ids,   chosen_mask   = self._encode_pair(sample['chosen'])
        rejected_ids, rejected_mask = self._encode_pair(sample['rejected'])
        # Same shift as dataset/lm_dataset.py:DPODataset.__getitem__.
        x_chosen   = torch.tensor(chosen_ids[:-1],    dtype=torch.long)
        y_chosen   = torch.tensor(chosen_ids[1:],     dtype=torch.long)
        m_chosen   = torch.tensor(chosen_mask[1:],    dtype=torch.long)
        x_rejected = torch.tensor(rejected_ids[:-1],  dtype=torch.long)
        y_rejected = torch.tensor(rejected_ids[1:],   dtype=torch.long)
        m_rejected = torch.tensor(rejected_mask[1:],  dtype=torch.long)
        return {
            'x_chosen': x_chosen, 'y_chosen': y_chosen, 'mask_chosen': m_chosen,
            'x_rejected': x_rejected, 'y_rejected': y_rejected, 'mask_rejected': m_rejected,
        }

Build a slice of `dpo.jsonl` so the demo stays fast on Colab. The `select(range(N))` upper bound and `dpo_seq_len` below are chosen so that:

* **`dpo_seq_len` covers the assistant p95 from Step 1.** With the real BPE tokenizer, `dpo_seq_len=32` would have truncated almost every assistant turn — the DPO loss mask would have been mostly zero and the loss would not move (the classic *DPO loss stuck at log 2 ≈ 0.693* symptom). A few hundred tokens is enough to cover the bulk of the chosen / rejected tokens and let the gradient flow.
* **`select(range(2000))`** keeps the demo runtime to a few minutes on a T4 GPU; bump this up for a real run.

In [23]:
# Build the dataset over the first slice of dpo.jsonl so the demo stays fast.
# Note: dpo_seq_len is now sized to fit the assistant token span (see Step 1
# stats); a 32-token window truncates the supervised region away and zeros out
# the loss mask, which is the most common reason the DPO loss does not move.
dpo_seq_len = 256
ds_dpo = DPODataset("./dataset/dpo.jsonl", tokenizer, max_length=dpo_seq_len)
ds_dpo.samples = ds_dpo.samples.select(range(min(2000, len(ds_dpo.samples))))
loader_dpo = DataLoader(ds_dpo, batch_size=4, shuffle=True)
print(f"DPO dataset: {len(ds_dpo)} preference pairs, max_seq_len={dpo_seq_len}")

# Quick sanity check on a single batch's shapes + assistant-mask coverage.
sample_batch = next(iter(loader_dpo))
for k, v in sample_batch.items():
    print(f"  {k:14s}: shape={tuple(v.shape)}, dtype={v.dtype}")
mask_cov = (sample_batch['mask_chosen'].sum() + sample_batch['mask_rejected'].sum()).item()
mask_tot = sample_batch['mask_chosen'].numel() + sample_batch['mask_rejected'].numel()
print(f"  assistant-token coverage in batch: {mask_cov}/{mask_tot} = {100.0*mask_cov/mask_tot:.1f}%")
print("  (must be > 0; if it is ~0, increase dpo_seq_len so the assistant span is not truncated.)")


Generating train split: 0 examples [00:00, ? examples/s]

DPO dataset: 2000 preference pairs, max_seq_len=256
  x_chosen      : shape=(4, 255), dtype=torch.int64
  y_chosen      : shape=(4, 255), dtype=torch.int64
  mask_chosen   : shape=(4, 255), dtype=torch.int64
  x_rejected    : shape=(4, 255), dtype=torch.int64
  y_rejected    : shape=(4, 255), dtype=torch.int64
  mask_rejected : shape=(4, 255), dtype=torch.int64
  assistant-token coverage in batch: 1531/2040 = 75.0%
  (must be > 0; if it is ~0, increase dpo_seq_len so the assistant span is not truncated.)


## Step 7: DPO Loss and Training Loop



### Theory & Concepts

#### From RLHF to DPO

Classical RLHF (used in InstructGPT/ChatGPT) is a three-stage pipeline:

1. **SFT** — supervised fine-tuning to produce a starting policy $\pi_{\text{ref}}$.
2. **Reward model** — train a separate model $r_\phi(x, y)$ on preference data via the Bradley–Terry likelihood $P(y_w \succ y_l) = \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))$.
3. **PPO** — optimise the policy $\pi_\theta$ to maximise $\mathbb{E}[r_\phi(x, y)] - \beta\,\mathrm{KL}[\pi_\theta\,\|\,\pi_{\text{ref}}]$, using on-policy roll-outs and the PPO clipping trick.

This works but has many moving parts (a separate reward model, a value head, on-policy sampling, careful PPO hyper-parameters). It is also expensive: each policy update requires fresh rollouts.

[Rafailov et al. 2023](https://arxiv.org/abs/2305.18290) showed that the **closed-form solution** of the KL-regularised RL objective in stage 3 is

$$\pi^*(y\,|\,x) \;\propto\; \pi_{\text{ref}}(y\,|\,x)\,\exp\!\left(\tfrac{1}{\beta}\,r(x,y)\right).$$

Inverting this gives an **implicit reward** in terms of any policy and the reference,

$$r(x,y) \;=\; \beta\,\log\!\frac{\pi_\theta(y\,|\,x)}{\pi_{\text{ref}}(y\,|\,x)} \;+\; \beta\,\log Z(x).$$

Plugging this into the Bradley–Terry preference likelihood, the partition function $Z(x)$ cancels, leaving the **DPO loss**:

$$\boxed{\;\mathscr{L}_{\text{DPO}}(\pi_\theta;\pi_{\text{ref}}) \;=\; -\,\mathbb{E}_{(x, y_w, y_l) \sim \mathscr{D}}\Bigg[\log\sigma\!\Big(\beta\,\big[\log\tfrac{\pi_\theta(y_w\,|\,x)}{\pi_{\text{ref}}(y_w\,|\,x)} \;-\; \log\tfrac{\pi_\theta(y_l\,|\,x)}{\pi_{\text{ref}}(y_l\,|\,x)}\big]\Big)\Bigg]\;}$$

DPO is therefore a single supervised loss — **no separate reward model, no PPO, no rollouts** — yet it is provably optimising the same objective as classical RLHF. That is why it has become the most popular post-training alignment recipe outside the largest labs.


#### Reading the formula

* The argument of $\sigma$ is the model's *log-likelihood-ratio margin* between chosen and rejected responses, normalised against the reference.
* If $\pi_\theta$ already prefers $y_w$ over $y_l$ *more strongly than the reference does*, the margin is positive, $\sigma(\beta\cdot\text{margin}) \to 1$, and $-\log\sigma \to 0$ — the loss is small.
* If the policy prefers $y_l$, the margin is negative and the loss grows.
* Crucially, the loss is *only* a function of how the policy's log-prob ratio differs from the reference's; it does not push absolute likelihoods.



#### Why subtract $\log\pi_{\text{ref}}$?

The reference term is the algebraic image of the KL penalty $\beta\,\mathrm{KL}[\pi_\theta\,\|\,\pi_{\text{ref}}]$ from RLHF. It prevents the policy from drifting too far from the SFT distribution and degenerating into mode collapse — a known failure mode of unconstrained reward maximisation.

#### The role of $\beta$

| $\beta$ | Effect |
|---|---|
| Very small ($10^{-9}$ … $10^{-7}$) | Conservative — policy barely moves from $\pi_{\text{ref}}$, easy to over-train, low impact |
| Moderate ($\sim 0.1$) | Typical setting — meaningful preference signal while still respecting SFT |
| Large ($> 1$) | Aggressive — policy can drift far from $\pi_{\text{ref}}$, may collapse into degenerate "preferred" tokens |

`trainer/train_dpo.py` defaults to `beta=0.1` and a tiny LR (`4e-8`); we use `beta=0.15` and a slightly larger LR so the demo's loss visibly moves on the small data slice.



#### The reference model

$\pi_{\text{ref}}$ is a **frozen, untrained deep-copy of the SFT model**. We never call `.backward()` on it; `requires_grad_(False)` plus `eval()` ensure no gradient or BatchNorm/Dropout state leaks in. In production, holding two copies of the model in GPU memory is the dominant cost of DPO; tricks like LoRA-DPO (only the LoRA delta is trainable, the reference is just the base model) reduce this dramatically.

### Implementation helpers

`trainer/train_dpo.py` is split into two helpers, both reproduced verbatim below:

| Helper | Inputs | Output |
|---|---|---|
| `logits_to_log_probs(logits, labels)` | `(B, T, V)` logits + `(B, T)` gold labels | `(B, T)` per-token log-probs of the gold labels |
| `dpo_loss(ref_log_probs, policy_log_probs, mask, beta)` | per-token log-probs from policy and reference + assistant-only mask + $\beta$ | scalar DPO loss |

Note that the trainer concatenates `chosen` and `rejected` along the **batch** dimension before the forward pass — `dpo_loss` then splits the doubled batch back in half along dim 0.

In [24]:
# ---- trainer/train_dpo.py (verbatim) ----------------------------------
def logits_to_log_probs(logits, labels):
    log_probs = F.log_softmax(logits, dim=2)
    return torch.gather(log_probs, dim=2, index=labels.unsqueeze(2)).squeeze(-1)

def dpo_loss(ref_log_probs, policy_log_probs, mask, beta):
    # Sum log-probabilities only over assistant tokens.
    ref_log_probs    = (ref_log_probs    * mask).sum(dim=1)
    policy_log_probs = (policy_log_probs * mask).sum(dim=1)
    # The trainer below stacks chosen + rejected along dim 0; split them here.
    bsz = ref_log_probs.shape[0]
    chosen_ref,    reject_ref    = ref_log_probs[:bsz // 2],    ref_log_probs[bsz // 2:]
    chosen_policy, reject_policy = policy_log_probs[:bsz // 2], policy_log_probs[bsz // 2:]
    pi_logratios  = chosen_policy - reject_policy
    ref_logratios = chosen_ref    - reject_ref
    logits = pi_logratios - ref_logratios
    return -F.logsigmoid(beta * logits).mean()

### DPO trainer

The structure mirrors `trainer/train_dpo.py:train_epoch`:

1. **Concatenate `chosen` and `rejected` along the batch dimension** so we can do a single forward pass through the policy and a single forward pass through the reference model.
2. Run `model(x)` to get `policy_logits`, then `logits_to_log_probs(policy_logits, y)` to get per-token log-probs of the gold targets.
3. Run `ref_model(x)` under `torch.no_grad()` and compute the same log-probs (the reference is frozen).
4. Apply `dpo_loss` with `beta=0.1` (the `train_dpo.py` default) and step the optimiser every batch.

> The original repo's `outputs.logits` / `outputs.aux_loss` come from the production `MiniMindForCausalLM` which returns a HuggingFace-style dataclass. Our micro model returns the `(logits, loss)` tuple, so we simply unpack it. There is no `aux_loss` term in this dense demo model.

In [25]:
# Build the DPO policy + reference model from the SFT (non-RLHF) checkpoint
# you provided in Step 3. trainer/train_dpo.py does the same: it init_model()s
# both `model` (policy) and `ref_model` (frozen) from `--model_path` and freezes
# the latter.
model_dpo = load_minimind_checkpoint(SFT_CKPT_PATH)

ref_model = load_minimind_checkpoint(SFT_CKPT_PATH)
ref_model.eval()
ref_model.requires_grad_(False)

# Position embeddings for the (now longer) DPO sequence length.
pos_embs_dpo = precompute_freqs_cis(config.head_dim, dpo_seq_len)
pos_embs_dpo = (pos_embs_dpo[0].to(args.device), pos_embs_dpo[1].to(args.device))

# Demo-only DPO hyper-parameters — see the table above for justification.
# Production should use train_dpo.py defaults (lr<=5e-8, beta=0.1).
dpo_lr                 = 1e-5
dpo_epochs             = 3
dpo_accumulation_steps = 1
beta                   = 0.1

opt_dpo    = optim.AdamW(model_dpo.parameters(), lr=dpo_lr)
scaler_dpo = torch.amp.GradScaler("cuda", enabled=("cuda" in args.device))
device_type  = "cuda" if "cuda" in args.device else "cpu"
autocast_ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast('cuda', dtype=torch.float16)

model_dpo.train()
print("\n>>> Starting Phase: DPO (RLHF) <<<")
print(f"    lr={dpo_lr}, epochs={dpo_epochs}, beta={beta}, accumulation_steps={dpo_accumulation_steps}, seq_len={dpo_seq_len}")
loss_history = []
for epoch in range(1, dpo_epochs + 1):
    last_step = 0
    for step, batch in enumerate(loader_dpo, start=1):
        last_step = step
        # Stack chosen + rejected along dim 0 — same trick as train_dpo.py:64-66.
        x = torch.cat([batch['x_chosen'],    batch['x_rejected']],    dim=0).to(args.device)
        y = torch.cat([batch['y_chosen'],    batch['y_rejected']],    dim=0).to(args.device)
        m = torch.cat([batch['mask_chosen'], batch['mask_rejected']], dim=0).to(args.device)
        cur_len = x.size(1)
        pe = (pos_embs_dpo[0][:cur_len], pos_embs_dpo[1][:cur_len])
        with autocast_ctx:
            with torch.no_grad():
                ref_logits, _ = ref_model(x, position_embeddings=pe)
                ref_log_probs = logits_to_log_probs(ref_logits, y)
            policy_logits, _ = model_dpo(x, position_embeddings=pe)
            policy_log_probs = logits_to_log_probs(policy_logits, y)
            loss = dpo_loss(ref_log_probs, policy_log_probs, m, beta=beta)
            loss = loss / dpo_accumulation_steps
        scaler_dpo.scale(loss).backward()
        if step % dpo_accumulation_steps == 0:
            scaler_dpo.unscale_(opt_dpo)
            torch.nn.utils.clip_grad_norm_(model_dpo.parameters(), args.grad_clip)
            scaler_dpo.step(opt_dpo); scaler_dpo.update()
            opt_dpo.zero_grad(set_to_none=True)
        if step % args.log_interval == 0:
            cur = loss.item() * dpo_accumulation_steps
            loss_history.append(cur)
            print(f"[DPO] Epoch {epoch}/{dpo_epochs} Step {step}: dpo_loss = {cur:.4f}")
    if last_step and last_step % dpo_accumulation_steps != 0:
        scaler_dpo.unscale_(opt_dpo)
        torch.nn.utils.clip_grad_norm_(model_dpo.parameters(), args.grad_clip)
        scaler_dpo.step(opt_dpo); scaler_dpo.update()
        opt_dpo.zero_grad(set_to_none=True)

if loss_history:
    head = sum(loss_history[: max(1, len(loss_history)//5)]) / max(1, len(loss_history)//5)
    tail = sum(loss_history[-max(1, len(loss_history)//5):]) / max(1, len(loss_history)//5)
    print(f"\n[DPO] mean loss over first 20% of logged steps: {head:.4f}")
    print(f"[DPO] mean loss over last  20% of logged steps: {tail:.4f}")
    print(f"[DPO] reduction: {head - tail:+.4f}  (positive = loss is decreasing)")

torch.save(model_dpo.state_dict(), "./checkpoints/dpo_model.pth")
print("Saved DPO checkpoint to ./checkpoints/dpo_model.pth")



>>> Starting Phase: DPO (RLHF) <<<
    lr=1e-05, epochs=3, beta=0.1, accumulation_steps=1, seq_len=256
[DPO] Epoch 1/3 Step 5: dpo_loss = 0.6751
[DPO] Epoch 1/3 Step 10: dpo_loss = 0.7116
[DPO] Epoch 1/3 Step 15: dpo_loss = 0.7394
[DPO] Epoch 1/3 Step 20: dpo_loss = 0.7004
[DPO] Epoch 1/3 Step 25: dpo_loss = 0.7287
[DPO] Epoch 1/3 Step 30: dpo_loss = 0.6305
[DPO] Epoch 1/3 Step 35: dpo_loss = 0.6554
[DPO] Epoch 1/3 Step 40: dpo_loss = 0.7196
[DPO] Epoch 1/3 Step 45: dpo_loss = 0.4816
[DPO] Epoch 1/3 Step 50: dpo_loss = 0.7277
[DPO] Epoch 1/3 Step 55: dpo_loss = 0.7336
[DPO] Epoch 1/3 Step 60: dpo_loss = 0.6492
[DPO] Epoch 1/3 Step 65: dpo_loss = 0.4796
[DPO] Epoch 1/3 Step 70: dpo_loss = 0.6714
[DPO] Epoch 1/3 Step 75: dpo_loss = 0.3750
[DPO] Epoch 1/3 Step 80: dpo_loss = 0.3763
[DPO] Epoch 1/3 Step 85: dpo_loss = 0.7408
[DPO] Epoch 1/3 Step 90: dpo_loss = 0.5014
[DPO] Epoch 1/3 Step 95: dpo_loss = 0.6297
[DPO] Epoch 1/3 Step 100: dpo_loss = 0.7903
[DPO] Epoch 1/3 Step 105: dpo_loss =

## Step 8: PPO — Proximal Policy Optimization (RLHF)



### Theory & Concepts


#### What problem does PPO solve?

PPO is the *original* RLHF policy-optimisation algorithm — the third stage of the InstructGPT/ChatGPT pipeline ([Schulman et al. 2017](https://arxiv.org/abs/1707.06347), [Ouyang et al. 2022](https://arxiv.org/abs/2203.02155)). After SFT and reward-model training, PPO fine-tunes the policy $\pi_\theta$ to maximise the KL-regularised reward objective

$$\max_\theta\; \mathbb{E}_{x \sim \mathscr{D},\, y \sim \pi_\theta(\cdot|x)}\Big[r_\phi(x, y)\Big] \;-\; \beta\,\mathrm{KL}\!\big[\pi_\theta(\cdot|x)\,\|\,\pi_{\text{ref}}(\cdot|x)\big].$$

The reward $r_\phi$ comes from a learned reward model (or a heuristic — see below); the KL term anchors $\pi_\theta$ to the SFT distribution $\pi_{\text{ref}}$ so it does not collapse into degenerate "reward-hacking" outputs.

Unlike DPO, PPO is **on-policy**: at every iteration we *sample new responses* from the current policy, score them, and use those rollouts as training data. This is more expressive than DPO (the policy can discover good responses that were never in the preference dataset) but also dramatically more expensive — every gradient step requires fresh autoregressive generation, a critic forward pass, and a frozen-reference forward pass.


#### Architectural ingredients

| Component | Symbol | Role | Source |
|---|---|---|---|
| **Actor / Policy** | $\pi_\theta$ | The language model being trained — produces token distributions, generates rollouts. | `trainer/train_ppo.py:actor_model` |
| **Critic** | $V_\psi(s_t)$ | A separate head over hidden states predicting the value (expected return) of every state. Used for variance reduction via GAE. | `CriticModel` in `trainer/train_ppo.py:36-48` (subclass of `MiniMindForCausalLM` with a `value_head: Linear(hidden, 1)`). |
| **Reference model** | $\pi_{\text{ref}}$ | A frozen copy of the SFT checkpoint. Used for the KL penalty inside the loss. | `ref_model` in `trainer/train_ppo.py` |
| **Reward function** | $r_\phi$ | Scores rollouts. The official PPO uses an external reward model (`LMForRewardModel`) plus heuristic shaping (length, `<think>` formatting, repetition penalty). | `calculate_rewards` in `trainer/train_ppo.py:51-75` |


#### The PPO loss (clipped surrogate)

PPO never optimises the raw policy gradient directly. Instead, after rolling out $y \sim \pi_{\theta_{\text{old}}}$, it imports the **importance ratio** $\rho_t = \pi_\theta(y_t|x, y_{<t}) / \pi_{\theta_{\text{old}}}(y_t|x, y_{<t})$ and a per-token advantage $\hat{A}_t$ computed by **Generalised Advantage Estimation (GAE)** ([Schulman et al. 2015](https://arxiv.org/abs/1506.02438)):

$$\hat{A}_t \;=\; \sum_{l=0}^{T-t-1} (\gamma\lambda)^l\,\delta_{t+l},\qquad \delta_t \;=\; r_t + \gamma\,V_\psi(s_{t+1}) - V_\psi(s_t).$$

The actor loss is the **clipped surrogate** that keeps $\pi_\theta$ inside a trust region around $\pi_{\theta_{\text{old}}}$:

$$\mathscr{L}^{\text{policy}}(\theta) \;=\; -\mathbb{E}_t\Big[\min\!\big(\rho_t\,\hat{A}_t,\; \mathrm{clip}(\rho_t,\,1-\epsilon,\,1+\epsilon)\,\hat{A}_t\big)\Big].$$

In addition, an explicit **KL-to-reference penalty** stops the policy from drifting from $\pi_{\text{ref}}$ across many gradient steps:

$$\mathscr{L}^{\text{kl}}(\theta) \;=\; \mathbb{E}_t\Big[\exp\big(\log\pi_{\text{ref}} - \log\pi_\theta\big) - (\log\pi_{\text{ref}} - \log\pi_\theta) - 1\Big]\quad(\text{the unbiased }k_3\text{ KL estimator}).$$

The **critic** is trained with a clipped value loss ($V$-trace style) to minimise the squared error against the GAE returns $R_t = \hat{A}_t + V_\psi(s_t)$:

$$\mathscr{L}^{\text{value}}(\psi) \;=\; \tfrac{1}{2}\,\mathbb{E}_t\Big[\max\!\big((V_\psi - R_t)^2,\; (\mathrm{clip}(V_\psi, V_{\text{old}}-\epsilon_v, V_{\text{old}}+\epsilon_v) - R_t)^2\big)\Big].$$

The total loss combines all three terms (mirroring `trainer/train_ppo.py:203-219`):

$$\mathscr{L}^{\text{PPO}} \;=\; \mathscr{L}^{\text{policy}} \;+\; \alpha_{\text{kl}}\,\mathscr{L}^{\text{kl}} \;+\; c_v\,\mathscr{L}^{\text{value}}.$$



#### Why all the clipping?

PPO's clipping is *the* trick that makes it stable. Vanilla policy gradient explodes when a single rollout has a large advantage and the new policy assigns it a much higher probability than the old one ($\rho_t \gg 1$). Clipping caps the per-step update at a small trust-region radius $\epsilon$ (typically $0.1{-}0.2$). The early-stop on KL (`approx_kl > early_stop_kl` in `train_ppo.py:195-196`) is a second safety net — if the policy is moving too fast within an iteration, we abort the inner PPO epoch.



#### Hyper-parameters that matter

| Symbol | Code | Typical | Effect |
|---|---|---|---|
| $\gamma$ | `args.gamma` | $1.0$ for short responses, $0.95-0.99$ otherwise | Discount factor in GAE. |
| $\lambda$ | `args.lam` | $0.95$ | Bias-variance trade-off in GAE. Higher = lower bias, higher variance. |
| $\epsilon$ | `args.clip_epsilon` | $0.2$ | Policy clip radius. |
| $\epsilon_v$ | `args.cliprange_value` | $0.2$ | Value-function clip radius. |
| $c_v$ | `args.vf_coef` | $0.1-1.0$ | Value-loss weight. |
| $\alpha_{\text{kl}}$ | `args.kl_coef` | $\sim 0.01$ | KL-to-reference penalty weight. |
| early-stop KL | `args.early_stop_kl` | $\sim 0.02$ | Hard cap on per-iteration drift. |
| inner epochs | `args.ppo_update_iters` | $2-4$ | How many gradient passes over each batch of rollouts. |



#### DPO vs. PPO — when to pick which?

| Property | DPO | PPO |
|---|---|---|
| Data required | Offline preference pairs `(chosen, rejected)` | Prompts only + a reward signal (model or heuristic) |
| Training samples | Every batch is a fresh preference pair | Every batch requires generating new rollouts |
| Compute cost | Two forward passes (policy + ref) per pair | Generation + actor + critic + ref forward passes |
| Implementation | One supervised loss | Rollout loop, GAE, critic, clipping, early-stop |
| Performance ceiling | Capped by quality of preference data | Higher — policy can discover novel high-reward outputs |
| Failure modes | Forgets if $\beta$ is too high; reward hacking via "shortcut" tokens | Mode collapse, reward hacking, value-function divergence |

In short: DPO is the cheaper, simpler default. PPO is the more powerful but more delicate option, and is still the standard at the largest labs.

### Implementation: a minimal PPO that mirrors `trainer/train_ppo.py`

Faithfully reproducing the full `trainer/train_ppo.py` pipeline (vLLM/SGLang rollout engine, `LMForRewardModel`, distributed training) is out of scope for a Colab demo. What follows is the *single-process, no-external-reward-model* version of the same algorithm — the structure is identical, only the rollout/reward primitives are simplified:

| Production (`trainer/train_ppo.py`) | This demo |
|---|---|
| `RolloutEngine` (vLLM / SGLang) | Plain autoregressive sampling with the in-memory actor (`generate_rollout`) |
| External `LMForRewardModel` + heuristic shaping (`calculate_rewards`) | Heuristic-only `compute_rewards` (length window, repetition penalty) |
| `DistributedDataParallel` actor + critic | Single-process actor + critic |
| All other steps (GAE, clipping, KL, early stop, value-clipping, inner PPO epochs) | **Bit-for-bit the same logic, copied from `trainer/train_ppo.py:114-251`** |

The four building blocks below are the only things you need to read carefully — the training loop after that is straight `train_ppo.py`.

In [26]:
# ---- CriticModel — mirrors trainer/train_ppo.py:36-48 -------------------
class CriticModel(MiniMindForCausalLM):
    """Reuse the policy backbone but replace the LM head with a scalar value head."""
    def __init__(self, params):
        super().__init__(params)
        self.value_head = nn.Linear(params.hidden_size, 1)

    def forward(self, input_ids, position_embeddings):
        # Forward through the policy backbone (`self.model`), ignore the LM
        # head, and project the final hidden states down to a per-token scalar.
        h = self.model(input_ids, position_embeddings)
        return self.value_head(h).squeeze(-1)  # [B, T]


In [27]:
# ---- Heuristic reward — slimmed-down trainer/train_ppo.py:51-75 ---------
# In production this delegates to LMForRewardModel.get_score(...). For the
# Colab demo we use only the deterministic heuristics (length window + 3-gram
# repetition penalty) so the notebook runs without an external reward model.

def repetition_penalty(token_ids, n=3, cap=0.5):
    if len(token_ids) < n:
        return 0.0
    grams = [tuple(token_ids[i:i + n]) for i in range(len(token_ids) - n + 1)]
    if not grams:
        return 0.0
    return min(cap, (len(grams) - len(set(grams))) * cap * 2 / len(grams))

def compute_rewards(prompt_ids, response_ids):
    """Return a [B] tensor of scalar rewards. Same shaping signs as train_ppo.py."""
    rewards = torch.zeros(len(response_ids), device=args.device)
    for i, resp in enumerate(response_ids):
        # Drop padding / eos so the heuristic looks at the real generation.
        clean = [t for t in resp if t not in (tokenizer.pad_token_id, tokenizer.eos_token_id)]
        L = len(clean)
        # Length window: discourage trivial or run-away responses.
        rewards[i] += 0.5 if 5 <= L <= 30 else -0.5
        # Repetition penalty.
        rewards[i] -= repetition_penalty(clean)
    return rewards

In [28]:
# ---- Rollout — replaces trainer/rollout_engine.create_rollout_engine -----
@torch.no_grad()
def generate_rollout(model, prompt_ids, pe_full, max_new_tokens=8, temperature=0.8):
    """Sample a continuation from `model` with multinomial sampling.

    Returns:
      gen_out:    [B, P + max_new_tokens]  — prompt followed by the rollout
      resp_ids:   list[list[int]] of length B — the generated tokens only
    """
    model.eval()
    out = prompt_ids.clone()
    for _ in range(max_new_tokens):
        cur_len = out.size(1)
        pe = (pe_full[0][:cur_len], pe_full[1][:cur_len])
        logits, _ = model(out, position_embeddings=pe)
        next_logits = logits[:, -1, :] / max(temperature, 1e-6)
        probs = F.softmax(next_logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1)
        out = torch.cat([out, next_tok], dim=1)
    resp_ids = [out[i, prompt_ids.size(1):].tolist() for i in range(out.size(0))]
    model.train()
    return out, resp_ids

In [29]:
# ---- PPO args — names mirror trainer/train_ppo.py argparse defaults -----
class PPOArgs:
    epochs                = 2     # was 1 — a single inner loop over 64 prompts barely moves the policy
    batch_size            = 4
    mini_batch_size       = 4
    accumulation_steps    = 1
    grad_clip             = 1.0
    actor_lr              = 5e-7
    critic_lr             = 1e-5
    gamma                 = 1.0
    lam                   = 0.95
    clip_epsilon          = 0.2
    cliprange_value       = 0.2
    vf_coef               = 0.5
    kl_coef               = 0.01
    early_stop_kl         = 0.02
    ppo_update_iters      = 3     # inner PPO epochs per batch (was 2)
    max_gen_len           = 12    # tokens to sample per rollout (was 8)
    log_interval          = 5

ppo_args = PPOArgs()


In [30]:
# Build a small RLAIF-style prompt loader. `RLAIFDataset` in dataset/lm_dataset.py
# emits prompts only; for the Colab demo we just slice the user prompts out of
# `sft_t2t_mini.jsonl` and tokenise them to a fixed length.
class PromptOnlyDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_length=16, n=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.prompts = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                if len(self.prompts) >= n: break
                rec = json.loads(line)
                for turn in rec.get('conversations', []):
                    if turn.get('role') == 'user' and turn.get('content'):
                        self.prompts.append(str(turn['content'])); break
    def __len__(self): return len(self.prompts)
    def __getitem__(self, i):
        ids = self.tokenizer(self.prompts[i], max_length=self.max_length).input_ids
        ids = ids + [self.tokenizer.pad_token_id] * (self.max_length - len(ids))
        return torch.tensor(ids[:self.max_length], dtype=torch.long)

ppo_prompt_len = 16
ppo_seq_len    = ppo_prompt_len + ppo_args.max_gen_len
ds_ppo = PromptOnlyDataset("./dataset/sft_t2t_mini.jsonl", tokenizer, max_length=ppo_prompt_len, n=128)
loader_ppo = DataLoader(ds_ppo, batch_size=ppo_args.batch_size, shuffle=True)
print(f"PPO prompt dataset: {len(ds_ppo)} prompts, prompt_len={ppo_prompt_len}, max_gen_len={ppo_args.max_gen_len}")


PPO prompt dataset: 128 prompts, prompt_len=16, max_gen_len=12


### PPO trainer

Mirrors the body of `trainer/train_ppo.py:ppo_train_epoch` (lines ~78–251):

1. **Build actor + critic + reference** from the SFT checkpoint provided in Step 3. The reference is frozen.
2. **Roll out** a batch of responses from the actor (`generate_rollout`).
3. **Score** the rollouts with `compute_rewards` (heuristic only in this demo).
4. **Compute old log-probs and old values** under `torch.no_grad()` — the "old policy" needed for importance sampling.
5. **GAE** loop: $\delta_t = r_t + \gamma V_{t+1} - V_t$, $\hat{A}_t = \delta_t + \gamma\lambda\hat{A}_{t+1}$. Returns are $R_t = \hat{A}_t + V_t$. Advantages are normalised across the batch (`train_ppo.py:155-157`).
6. **Inner PPO epochs**: for each mini-batch, recompute new log-probs and values, and minimise the **clipped surrogate + clipped value loss + KL-to-reference** (`train_ppo.py:170-241`). Early-stop the inner loop if `approx_kl > early_stop_kl`.
7. **Save** `ppo_model.pth`.

In [31]:
# ---- PPO training loop — structure copied from trainer/train_ppo.py ------
actor_model  = load_minimind_checkpoint(SFT_CKPT_PATH)
ref_model_ppo = load_minimind_checkpoint(SFT_CKPT_PATH)
ref_model_ppo.eval(); ref_model_ppo.requires_grad_(False)

# Critic: same backbone (initialised from the SFT weights via load_state_dict
# with strict=False — value_head is fresh).
critic_model = CriticModel(config).to(args.device)
critic_state = torch.load(SFT_CKPT_PATH, weights_only=True, map_location=args.device)
critic_model.load_state_dict(critic_state, strict=False)

actor_opt  = optim.AdamW(actor_model.parameters(),  lr=ppo_args.actor_lr)
critic_opt = optim.AdamW(critic_model.parameters(), lr=ppo_args.critic_lr)

# Position embeddings sized to fit prompt + max generation.
pe_full = precompute_freqs_cis(config.head_dim, ppo_seq_len + 4)
pe_full = (pe_full[0].to(args.device), pe_full[1].to(args.device))

print("\n>>> Starting Phase: PPO (RLHF) <<<")
grad_accum_step = 0
for epoch in range(1, ppo_args.epochs + 1):
    for step, prompt_ids in enumerate(loader_ppo, start=1):
        prompt_ids = prompt_ids.to(args.device)
        B          = prompt_ids.size(0)
        prompt_len = prompt_ids.size(1)

        # 1) Rollout under the *old* policy (gradients off).
        gen_out, resp_ids = generate_rollout(
            actor_model, prompt_ids, pe_full,
            max_new_tokens=ppo_args.max_gen_len, temperature=0.8,
        )
        rewards = compute_rewards([p.tolist() for p in prompt_ids], resp_ids)  # [B]

        labels       = gen_out[:, 1:].clone()                              # [B, T-1]
        seq_len_eff  = labels.size(1)
        resp_start   = prompt_len - 1                                      # response slice start
        cur_pe       = (pe_full[0][:seq_len_eff + 1], pe_full[1][:seq_len_eff + 1])

        # Mask of "valid response token" positions (no pad).
        resp_labels  = labels[:, resp_start:]                              # [B, R]
        resp_mask    = (resp_labels != tokenizer.pad_token_id).float()     # [B, R]

        # 2) old_log_probs, old_values, ref_log_probs — train_ppo.py:129-141.
        with torch.no_grad():
            old_values_seq = critic_model(gen_out, position_embeddings=cur_pe)
            old_resp_values = old_values_seq[:, resp_start:-1] * resp_mask    # [B, R]
            old_logits, _   = actor_model(gen_out, position_embeddings=cur_pe)
            old_logp_all    = F.log_softmax(old_logits[:, :-1], dim=-1).gather(2, labels.unsqueeze(-1)).squeeze(-1)
            old_resp_logp   = old_logp_all[:, resp_start:]                    # [B, R]
            ref_logits, _   = ref_model_ppo(gen_out, position_embeddings=cur_pe)
            ref_logp_all    = F.log_softmax(ref_logits[:, :-1], dim=-1).gather(2, labels.unsqueeze(-1)).squeeze(-1)
            ref_resp_logp   = ref_logp_all[:, resp_start:]                    # [B, R]

            # 3) Token-level rewards: terminal reward at the last response token
            #    (same shape as train_ppo.py:142-144).
            token_rewards = torch.zeros_like(old_resp_logp)
            last_idx      = (resp_mask.long().sum(dim=1) - 1).clamp(min=0)
            token_rewards[torch.arange(B, device=args.device), last_idx] += rewards

            # 4) GAE — train_ppo.py:146-153.
            R = old_resp_values.size(1)
            lastgae = torch.zeros(B, device=args.device); advs_rev = []
            for t in reversed(range(R)):
                nv    = old_resp_values[:, t + 1] if t < R - 1 else 0.0
                delta = token_rewards[:, t] + ppo_args.gamma * nv - old_resp_values[:, t]
                lastgae = delta + ppo_args.gamma * ppo_args.lam * lastgae
                advs_rev.append(lastgae)
            advantages = torch.stack(advs_rev[::-1], dim=1)                   # [B, R]
            returns    = advantages + old_resp_values

            # Normalise advantages across the response tokens (train_ppo.py:155-157).
            adv_mean = (advantages * resp_mask).sum() / resp_mask.sum().clamp(min=1)
            adv_var  = (((advantages - adv_mean) ** 2) * resp_mask).sum() / resp_mask.sum().clamp(min=1)
            advantages = (advantages - adv_mean) * torch.rsqrt(adv_var + 1e-8) * resp_mask

        # 5) Inner PPO epochs — train_ppo.py:170-241.
        stop_ppo = False
        mb_size = max(1, min(ppo_args.mini_batch_size, B))
        last_metrics = {}
        actor_model.train(); critic_model.train()
        for inner in range(ppo_args.ppo_update_iters):
            if stop_ppo: break
            order = torch.randperm(B, device=args.device)
            for off in range(0, B, mb_size):
                inds = order[off:off + mb_size]

                # New values + new log-probs under the *current* policy.
                mb_values_seq  = critic_model(gen_out[inds], position_embeddings=cur_pe)
                mb_resp_values = mb_values_seq[:, resp_start:-1]
                mb_logits, _   = actor_model(gen_out[inds], position_embeddings=cur_pe)
                mb_logp_all    = F.log_softmax(mb_logits[:, :-1], dim=-1).gather(2, labels[inds].unsqueeze(-1)).squeeze(-1)
                mb_resp_logp   = mb_logp_all[:, resp_start:]

                log_ratio = mb_resp_logp - old_resp_logp[inds]
                ratio     = torch.exp(log_ratio)
                approx_kl = (0.5 * (log_ratio ** 2) * resp_mask[inds]).sum() / resp_mask[inds].sum().clamp(min=1)
                if approx_kl.item() > ppo_args.early_stop_kl:
                    stop_ppo = True

                clipfrac = ((((ratio - 1.0).abs() > ppo_args.clip_epsilon).float() * resp_mask[inds]).sum()
                            / resp_mask[inds].sum().clamp(min=1))
                # k3 KL estimator vs the frozen reference (train_ppo.py:201-202).
                kl_ref_pen = ((torch.exp(ref_resp_logp[inds] - mb_resp_logp) - (ref_resp_logp[inds] - mb_resp_logp) - 1.0)
                              * resp_mask[inds]).sum() / resp_mask[inds].sum().clamp(min=1)

                # Clipped surrogate (train_ppo.py:203-206).
                policy_loss = ((torch.max(-advantages[inds] * ratio,
                                          -advantages[inds] * torch.clamp(ratio, 1.0 - ppo_args.clip_epsilon, 1.0 + ppo_args.clip_epsilon))
                               * resp_mask[inds]).sum() / resp_mask[inds].sum().clamp(min=1)
                               + ppo_args.kl_coef * kl_ref_pen)
                # Clipped value loss (train_ppo.py:207-210).
                value_loss  = 0.5 * (torch.max((mb_resp_values - returns[inds]) ** 2,
                                               (torch.clamp(mb_resp_values, old_resp_values[inds] - ppo_args.cliprange_value,
                                                            old_resp_values[inds] + ppo_args.cliprange_value) - returns[inds]) ** 2)
                                     * resp_mask[inds]).sum() / resp_mask[inds].sum().clamp(min=1)

                if stop_ppo:
                    loss = (policy_loss + ppo_args.vf_coef * value_loss) * 0.0  # keep DDP-safe forward/backward
                else:
                    loss = (policy_loss + ppo_args.vf_coef * value_loss) / ppo_args.accumulation_steps
                loss.backward()

                grad_accum_step += 1
                if grad_accum_step % ppo_args.accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(actor_model.parameters(),  ppo_args.grad_clip)
                    torch.nn.utils.clip_grad_norm_(critic_model.parameters(), ppo_args.grad_clip)
                    actor_opt.step();  actor_opt.zero_grad(set_to_none=True)
                    critic_opt.step(); critic_opt.zero_grad(set_to_none=True)

                last_metrics = {
                    'policy_loss': policy_loss.item(),
                    'value_loss':  value_loss.item(),
                    'kl':          approx_kl.item(),
                    'kl_ref':      kl_ref_pen.item(),
                    'clipfrac':    clipfrac.item(),
                    'reward':      rewards.mean().item(),
                }

        if step % ppo_args.log_interval == 0 and last_metrics:
            print(f"[PPO] Epoch {epoch}/{ppo_args.epochs} Step {step}: "
                  f"reward={last_metrics['reward']:+.3f} "
                  f"policy_loss={last_metrics['policy_loss']:+.4f} "
                  f"value_loss={last_metrics['value_loss']:.4f} "
                  f"kl={last_metrics['kl']:.4f} kl_ref={last_metrics['kl_ref']:.4f} "
                  f"clipfrac={last_metrics['clipfrac']:.3f}")

torch.save(actor_model.state_dict(), "./checkpoints/ppo_model.pth")
print("Saved PPO checkpoint to ./checkpoints/ppo_model.pth")


>>> Starting Phase: PPO (RLHF) <<<
[PPO] Epoch 1/2 Step 5: reward=+0.000 policy_loss=-0.0066 value_loss=0.0165 kl=0.0004 kl_ref=0.0018 clipfrac=0.000
[PPO] Epoch 1/2 Step 10: reward=+0.000 policy_loss=-0.0052 value_loss=0.0070 kl=0.0001 kl_ref=0.0029 clipfrac=0.000
[PPO] Epoch 1/2 Step 15: reward=+0.075 policy_loss=-0.0056 value_loss=0.0168 kl=0.0001 kl_ref=0.0018 clipfrac=0.000
[PPO] Epoch 1/2 Step 20: reward=+0.000 policy_loss=-0.0098 value_loss=0.0088 kl=0.0002 kl_ref=0.0066 clipfrac=0.000
[PPO] Epoch 1/2 Step 25: reward=+0.132 policy_loss=-0.0120 value_loss=0.0098 kl=0.0002 kl_ref=0.0115 clipfrac=0.000
[PPO] Epoch 1/2 Step 30: reward=+0.000 policy_loss=-0.0070 value_loss=0.0146 kl=0.0001 kl_ref=0.0023 clipfrac=0.000
[PPO] Epoch 2/2 Step 5: reward=+0.014 policy_loss=-0.0026 value_loss=0.0250 kl=0.0001 kl_ref=0.0046 clipfrac=0.000
[PPO] Epoch 2/2 Step 10: reward=+0.125 policy_loss=-0.0086 value_loss=0.0462 kl=0.0002 kl_ref=0.0065 clipfrac=0.000
[PPO] Epoch 2/2 Step 15: reward=+0.000

## Step 9: Side-by-Side Comparison — Pretrain vs SFT vs SFT+LoRA vs DPO vs PPO

We now have five checkpoints to compare:

| Stage | File | What it is |
|---|---|---|
| 0. Pretrain (no instruction tuning) | `PRETRAIN_CKPT_PATH` | The raw language model — has never seen a chat-format conversation |
| 1. SFT (no RLHF)                    | `SFT_CKPT_PATH` | Pretrain + supervised fine-tuning on instruction data — our base for everything else |
| 2. SFT + LoRA (merged)              | `./checkpoints/sft_lora_merged.pth` | LoRA adapter (trained on the chosen base) merged into the base weights |
| 3. DPO                              | `./checkpoints/dpo_model.pth` | RLHF via offline preference pairs (`dpo.jsonl`) |
| 4. PPO                              | `./checkpoints/ppo_model.pth` | RLHF via on-policy rollouts + clipped surrogate loss |

We load each into its own `MiniMindForCausalLM` instance and feed them all the same prompt with greedy decoding. Including **pretrain** and **SFT** as baselines is important: any qualitative shift attributed to LoRA / DPO / PPO must be interpreted *relative* to where the SFT model started, and the gap between pretrain and SFT shows what supervised instruction tuning alone already buys you.

> Both the demo subsets and the small number of post-training steps mean the visible difference between checkpoints will be subtle on a single prompt. The point is to show that **the same generation function**, fed the **same prompt** and using **five different checkpoints produced by five different training procedures**, returns five potentially different outputs. In a full run you would observe the well-known qualitative shifts (pretrain → no chat structure; SFT → follows the instruction; LoRA → domain-style replies; DPO/PPO → preference-aligned replies).

In [32]:
def generate_text(model, tokenizer, prompt, max_new_tokens=15):
    model.eval()
    input_ids = torch.tensor([tokenizer(prompt, max_length=args.max_seq_len).input_ids], dtype=torch.long).to(args.device)
    pe_full = precompute_freqs_cis(config.head_dim, input_ids.size(1) + max_new_tokens)
    pe_full = (pe_full[0].to(args.device), pe_full[1].to(args.device))
    with torch.no_grad():
        for _ in range(max_new_tokens):
            cur_len = input_ids.size(1)
            cur_pe = (pe_full[0][:cur_len], pe_full[1][:cur_len])
            logits, _ = model(input_ids, position_embeddings=cur_pe)
            next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
    return tokenizer.decode(input_ids[0].cpu().tolist())

# Load all five model variants — including the original pretrain and SFT baselines.
model_pretrain_eval = load_minimind_checkpoint(PRETRAIN_CKPT_PATH)
model_sft_eval      = load_minimind_checkpoint(SFT_CKPT_PATH)
model_lora_eval     = load_minimind_checkpoint("./checkpoints/sft_lora_merged.pth")
model_dpo_eval      = load_minimind_checkpoint("./checkpoints/dpo_model.pth")
model_ppo_eval      = load_minimind_checkpoint("./checkpoints/ppo_model.pth")

test_prompt = "Hello Assistant, can you help me?"
print(f"[Prompt]: {test_prompt}\n")

out_pre  = generate_text(model_pretrain_eval, tokenizer, test_prompt, max_new_tokens=10)
out_sft  = generate_text(model_sft_eval,      tokenizer, test_prompt, max_new_tokens=10)
out_lora = generate_text(model_lora_eval,     tokenizer, test_prompt, max_new_tokens=10)
out_dpo  = generate_text(model_dpo_eval,      tokenizer, test_prompt, max_new_tokens=10)
out_ppo  = generate_text(model_ppo_eval,      tokenizer, test_prompt, max_new_tokens=10)

print("=== STAGE-BY-STAGE COMPARISON ===")
print("0. Pretrain (no instruction tuning) Output:")
print(f"> {out_pre}\n")
print("1. SFT (no RLHF) Output:")
print(f"> {out_sft}\n")
print("2. SFT + LoRA (merged) Output:")
print(f"> {out_lora}\n")
print("3. DPO (RLHF, offline) Output:")
print(f"> {out_dpo}\n")
print("4. PPO (RLHF, on-policy) Output:")
print(f"> {out_ppo}\n")
print("=================================")


[Prompt]: Hello Assistant, can you help me?

=== STAGE-BY-STAGE COMPARISON ===
0. Pretrain (no instruction tuning) Output:
> Hello Assistant, can you help me?<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>

1. SFT (no RLHF) Output:
> Hello Assistant, can you help me?<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>

2. SFT + LoRA (merged) Output:
> Hello Assistant, can you help me?<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>

3. DPO (RLHF, offline) Output:
> Hello Assistant, can you help me?<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>

4. PPO (RLHF, on-policy) Output:
> Hello Assistant, can you help me?<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>�����



### What to look for in a real run

* **Pretrain vs SFT:** the pretrain model has never seen a chat-formatted conversation, so it tends to either continue the prompt as plain text or drift into unrelated webby content. The SFT model, by contrast, picks up the *instruction-following* shape (it tries to *answer* the question rather than continue it). This gap is the entire point of supervised fine-tuning.
* **SFT vs LoRA-merged:** the LoRA-merged model should reflect the style/domain of the LoRA adaptation data. With `lora_medical.jsonl` you would see domain-specific vocabulary in the response; with `lora_identity.jsonl` you would see the model claim its custom identity.
* **SFT vs DPO:** DPO-trained models tend to be more concise, refuse unsafe requests more cleanly, and pick the response style preferred by the annotators in `dpo.jsonl`. Because $\beta$ is small and the reference model regularises toward the SFT distribution, the change in surface form is usually subtle on individual prompts but compounds across an evaluation set.
* **SFT vs PPO:** PPO with our heuristic reward + small prompt set is unlikely to produce a dramatic shift, but it should still drift in the direction the reward function rewards (e.g. shorter, less repetitive answers if the heuristic penalises repetition).
* **Catastrophic forgetting:** if the DPO learning rate is too high (>5e-8 for the production model) you will see the DPO model drift away from coherent text and hallucinate more — which is why `train_dpo.py` defaults to `4e-8`. The demo bumps the LR to `1e-5` so the loss visibly moves on a 2 000-pair subset; this is fine for a teaching demo but should *not* be copied into production.

For full evaluation in production, see `eval_llm.py`, which loads `full_sft`, `dpo`, etc. and runs the same prompt suite through each weight.

## Wrap-up & Next Steps

In this notebook we built three of the most important post-training stages of a modern LLM pipeline from scratch, each anchored to the corresponding script under `trainer/`:

| Stage | Trainable params | Loss | Data | Reference file |
|---|---|---|---|---|
| **LoRA** | Only `A`/`B` adapters (~0.5% of base) | SFT cross-entropy | Domain-specific conversations | `trainer/train_lora.py` |
| **DPO** | All weights (typically) | $-\log\sigma(\beta\,\text{margin})$ | Preference pairs `(chosen, rejected)` | `trainer/train_dpo.py` |
| **PPO** | Actor + critic | Clipped surrogate + clipped value + KL-to-ref | Prompts + reward signal (heuristic or learned RM) | `trainer/train_ppo.py` |



### Where to go next

* **LoRA-DPO / LoRA-PPO** — combine the techniques: freeze the base, attach LoRA, train the LoRA delta with the DPO/PPO loss against a frozen reference (the base model itself). This is the cheapest known production-grade RLHF recipe.
* **Quantised base + LoRA (QLoRA)** — store the base model in 4-bit and only the LoRA adapter in fp16; lets you fine-tune 7B+ models on a single consumer GPU.
* **Plug a real reward model** into the PPO step (replace `compute_rewards` with `LMForRewardModel.get_score(...)` from `trainer/trainer_utils.py`) and use the production rollout engine (`trainer/rollout_engine.py`) for high-throughput generation.
* **GRPO** — a simpler critic-free variant of PPO; see `trainer/train_grpo.py`.
* **Full evaluation** — plug a real tokenizer and run `eval_llm.py` over the standard MiniMind eval prompts to see real qualitative differences between the four stages.
